# Lekcja 18: Zabezpieczanie agentów AI za pomocą kryptograficznych potwierdzeń

## Interaktywny notatnik

Ten notatnik prowadzi przez cztery ćwiczenia:

1. **Podpisz swoje pierwsze potwierdzenie** dla wywołania narzędzia agenta i je zweryfikuj.
2. **Zmodyfikuj potwierdzenie** i zobacz, jak weryfikacja zawodzi.
3. **Zbuduj łańcuch z trzech potwierdzeń** i potwierdź integralność łańcucha.
4. **Opakuj wywołanie narzędzia Microsoft Agent Framework** tak, aby każda akcja generowała potwierdzenie.

Wszystkie prymitywy kryptograficzne są importowane z dobrze utrzymywanych bibliotek (`pynacl` dla Ed25519, `jcs` dla RFC 8785 kanonicznego JSON, `hashlib` z biblioteki standardowej Pythona dla SHA-256). Sama logika potwierdzenia to czysty Python, który możesz czytać i modyfikować.

Uruchamiaj komórki w kolejności. Każda sekcja jest krótka i samodzielna.


## Instalacja

Zainstaluj dwa zależności. Obie mają licencje zezwalające na swobodne użytkowanie (Apache-2.0 / MIT).


In [ ]:
!pip install -q pynacl jcs

In [ ]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import BadSignatureError
from jcs import canonicalize

## Narzędzia pomocnicze

Te dwa pomocnicze narzędzia obsługują kodowanie base64url (bez wypełnienia) oraz haszowanie SHA-256 dowolnych obiektów. Pozwalają one zachować resztę notatnika skupioną na samej logice paragonu.


In [ ]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """Decode a base64url string that may be missing padding."""
    padding = "=" * ((4 - len(s) % 4) % 4)
    return base64.urlsafe_b64decode(s + padding)

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## Sekcja 1: Podpisz swój pierwszy paragon

Wyobraź sobie, że nasz agent dla **Contoso Travel** właśnie wyszukał loty z Sydney do Los Angeles dla klienta. Chcemy zarejestrować to wywołanie narzędzia jako podpisany paragon, aby przyszły audytor mógł to zweryfikować bez konieczności nam ufać.

### Krok 1.1: Wygeneruj klucz do podpisu

W środowisku produkcyjnym klucz podpisujący agenta znajdowałby się w module bezpieczeństwa sprzętowego (HSM), Azure Key Vault lub podobnym chronionym magazynie. W tej lekcji generujemy nowy klucz w pamięci.


In [ ]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

### Krok 1.2: Zbuduj ładunek potwierdzenia

Ładunek zawiera wszystko, co chcemy, aby potwierdzenie poświadczało: kto działał, jakim narzędziem, z jakimi argumentami, co zostało zwrócone, według jakiej polityki i kiedy. Haszujemy argumenty i wynik zamiast umieszczać je inline, aby potwierdzenie nie ujawniało poufnych treści.


In [ ]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

### Krok 1.3: Podpisz i zmontuj potwierdzenie

Trzy kroki:

1. Kanonizuj ładunek danych za pomocą JCS, aby dwie implementacje generujące to samo logiczne potwierdzenie produkowały identyczne bajty.
2. Oblicz skrót kanonicznych bajtów za pomocą SHA-256.
3. Podpisz skrót za pomocą prywatnego klucza Ed25519.

Następnie podpis jest dołączany do oryginalnego ładunku danych, tworząc ostateczne potwierdzenie.


In [ ]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()
    signature_bytes = signing_key.sign(message_hash).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

### Krok 1.4: Zweryfikuj pokwitowanie

Weryfikacja odwraca proces. Usuwamy podpis, ponownie obliczamy kanoniczny skrót i sprawdzamy podpis względem klucza publicznego w pokwitowaniu.

Audytor przeprowadzający tę weryfikację nie potrzebuje od nas niczego poza samym pokwitowaniem. Nie musi wywoływać żadnej usługi, nie musi pytać katalogu kluczy, nie wymaga żadnego zaufania.


In [ ]:
def verify_receipt(receipt: dict) -> bool:
    """
    Verify a receipt's Ed25519 signature.
    Returns True if valid, False otherwise.
    """
    sig_obj = receipt.get("signature")
    if not sig_obj or sig_obj.get("alg") != "EdDSA":
        return False

    # Reconstruct the payload that was actually signed (everything except signature).
    payload = {k: v for k, v in receipt.items() if k != "signature"}

    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()

    try:
        verify_key = signing.VerifyKey(b64url_decode(sig_obj["public_key"]))
        verify_key.verify(message_hash, b64url_decode(sig_obj["sig"]))
        return True
    except BadSignatureError:
        return False
    except Exception as exc:
        print(f"Verification error: {exc}")
        return False

is_valid = verify_receipt(receipt)
print(f"Receipt is valid: {is_valid}")

Powinieneś zobaczyć `Receipt is valid: True`. Agent wygenerował swój pierwszy kryptograficznie podpisany rekord audytu.


## Sekcja 2: Manipulowanie paragonem

Cały sens paragonów polega na tym, że są odporne na manipulacje. Udowodnijmy to.

Zmienimy dokładnie jeden znak na paragonie i zaobserwujemy, jak weryfikacja zawiedzie.


In [ ]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered)}")

### Co się właśnie stało?

Kiedy zmieniliśmy `policy_id`, zmieniły się kanoniczne bajty. Zmieniony został hash SHA-256 tych bajtów. Podpis (który obejmował oryginalny hash) już nie pasuje do nowego hasha. Weryfikacja poprawnie zwraca `False`.

Nie ma możliwości zmodyfikowania żadnego pola paragonu aby wciąż przeszedł on weryfikację, chyba że atakujący posiada klucz prywatny. Dopóki klucz prywatny jest w sejfie na klucze, a klucz publiczny został opublikowany, próby manipulacji są niemożliwe do ukrycia.

Spróbuj sam: zmodyfikuj `tool_name` lub `agent_id` albo `timestamp` w powyższej komórce i uruchom ponownie. Każda zmiana powoduje, że paragon jest nieważny.


## Sekcja 3: Łączenie potwierdzeń w łańcuch

Pojedyncze potwierdzenie chroni jedną akcję. Większość agentów wykonuje wiele akcji. Aby cały ciąg był odporny na manipulacje, łączymy każde potwierdzenie z poprzednim, umieszczając skrót poprzedniego potwierdzenia w ładunku nowego potwierdzenia.

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

Jeśli ktoś usunie lub zmieni kolejność potwierdzeń, łańcuch zostaje przerwany dokładnie w tym miejscu. Weryfikacja dowolnego późniejszego potwierdzenia kończy się niepowodzeniem, ponieważ jego `previous_receipt_hash` nie zgadza się już z rzeczywistym haszem poprzednika.


In [ ]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [ ]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

In [ ]:
def verify_chain(chain: list) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt (except the genesis) must reference the previous receipt's hash.
      3. Sequence numbers must match each receipt's zero-based position in the chain.
    Returns a list of per-receipt result dicts.
    """
    results = []
    for i, receipt in enumerate(chain):
        sig_ok = verify_receipt(receipt)

        if i == 0:
            chain_ok = receipt["previous_receipt_hash"] is None
        else:
            expected = receipt_hash(chain[i - 1])
            chain_ok = receipt["previous_receipt_hash"] == expected

        seq_ok = receipt["sequence"] == i

        results.append({
            "index": i,
            "tool": receipt["tool_name"],
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok,
        })
    return results

for r in verify_chain(chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

Teraz przerwij łańcuch, manipulując środkowym potwierdzeniem i ponownie zweryfikuj. Zmienione potwierdzenie nie przechodzi kontroli podpisu, A następne potwierdzenie nie przechodzi kontroli łańcucha (ponieważ jego `previous_receipt_hash` nie pasuje już do zmodyfikowanego hasza środkowego potwierdzenia).


In [ ]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

Potwierdzenie 0 nadal jest ważne (nie zostało zmodyfikowane i nie ma poprzednika, od którego byłoby zależne). Potwierdzenie 1 nie przechodzi kontroli podpisu, ponieważ zmieniliśmy `tool_args_hash`. Potwierdzenie 2 nie przechodzi kontroli łańcucha, ponieważ jego `previous_receipt_hash` zostało obliczone względem oryginalnego (obecnie zmodyfikowanego) potwierdzenia 1.

Nawet jeśli atakujący ponownie podpisze zmodyfikowane potwierdzenie 1 (czego nie może zrobić bez klucza prywatnego), rozbieżność w łańcuchu potwierdzenia 2 nadal ujawni manipulację. Aby ukryć zmianę, atakujący musiałby ponownie podpisać każde potwierdzenie od punktu modyfikacji wzwyż, co wymaga posiadania klucza prywatnego.


## Sekcja 4: Opakowanie wywołania narzędzia agenta z podpisem potwierdzenia

W rzeczywistej implementacji nie chcesz, aby każdy autor agenta pamiętał o wywołaniu `make_receipt`. Chcesz, aby podpisywanie potwierdzenia było automatyczne przy każdym wywołaniu narzędzia.

Oto najprostszy wzorzec: klasa opakowująca, która przyjmuje dowolną wywoływalną funkcję narzędzia i zwraca jej wersję emitującą potwierdzenie. To dostosowuje się do każdego frameworka agenta, w tym Microsoft Agent Framework (`agent_framework.foundry`).

Jeśli nie masz skonfigurowanego projektu Microsoft Foundry, poniższy lokalny mock nadal demonstruje ten wzorzec.


In [ ]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [ ]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to FoundryChatClient as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")


### Integracja z Microsoft Agent Framework

Powłoka `ReceiptedTool` powyżej jest niezależna od frameworka. Aby użyć jej w agencie zbudowanym przy użyciu Microsoft Agent Framework, zarejestruj owiniętą funkcję jako narzędzie. Szkic (należałoby zastąpić mock rzeczywistą rejestracją narzędzia Microsoft Foundry):

```python
# Pseudokod pokazujący kształt integracji.
# import os
# from agent_framework.foundry import FoundryChatClient
# from azure.identity import AzureCliCredential
#
# provider = FoundryChatClient(
#     project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
#     model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
#     credential=AzureCliCredential(),
# )
# agent = provider.as_agent(
#     instructions="Jesteś agentem podróży Contoso ...",
#     tools=[receipted_lookup],   # opakowane narzędzie, nie surowa funkcja
# )
# response = agent.run("Znajdź loty z Sydney do Los Angeles w czerwcu.")
#
# # Po wykonaniu, każde wywołanie narzędzia dokonane przez agenta ma podpisany pokwitowanie:
# audit_chain = receipted_lookup.receipts
```

Framework agenta nie musi nic wiedzieć o potwierdzeniach. Podpisywanie potwierdzeń jest owinięte wokół narzędzia, a nie wbudowane w framework. Tak dodajesz pochodzenie do istniejącego kodu agenta bez przepisywania agenta.


## Podsumowanie i wyzwanie rozciągające

Masz:

- Wygenerowaną parę kluczy Ed25519.
- Utworzony i podpisany dowód wywołania narzędzia agenta.
- Zweryfikowany dowód offline przy użyciu tylko klucza publicznego.
- Sfałszowany dowód i zaobserwowaną niepowodzenie weryfikacji.
- Zbudowaną sekwencję trzech dowodów połączonych skrótami.
- Sfałszowany środek łańcucha i zaobserwowane zarówno niepowodzenie podpisu, jak i niepowodzenie łącza łańcucha.
- Opakowaną funkcję narzędzia z automatycznym podpisywaniem dowodu.

**Wyzwanie rozciągające.** Rozszerz schemat dowodu o pole `request_id` (UUID do śledzenia rozproszonego). Zaktualizuj `make_receipt`, aby je uwzględnić, i potwierdź, że dowody nadal weryfikują się od końca do końca. Następnie zmodyfikuj pole po podpisaniu i potwierdź, że weryfikacja nie powiodła się. To zmusza do wewnętrznego zrozumienia, jak każdy bajt kanonicznego kodowania przyczynia się do podpisu.

**Ważna granica.** Dowody potwierdzają trzy rzeczy i tylko trzy: autorstwo (ten klucz podpisał tę treść), integralność (treść nie zmieniła się od podpisania) oraz kolejność (ten dowód nastąpił po tamtym dowodzie). Nie potwierdzają, że działanie agenta było poprawne, że faktycznie oceniono politykę nazwaną w `policy_id`, ani że agent przestrzegał każdej reguły. Dowody są fundamentem. Zarządzanie to system, który budujesz na ich bazie.

Przeczytaj ponownie README lekcji mając tę granicę na uwadze. Najczęstszym błędem zespołów związanym z dowodami jest założenie, że "mamy dowody" oznacza "jesteśmy zarządzani". Nie oznacza. Dowody czynią zachowanie agenta audytowalnym. Nie czynią go poprawnym.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Zastrzeżenie**:
Niniejszy dokument został przetłumaczony za pomocą usługi tłumaczenia AI [Co-op Translator](https://github.com/Azure/co-op-translator). Choć dążymy do dokładności, prosimy pamiętać, że automatyczne tłumaczenia mogą zawierać błędy lub niedokładności. Oryginalny dokument w jego języku źródłowym należy uznawać za autorytatywne źródło. W przypadku informacji krytycznych zalecane jest skorzystanie z profesjonalnego tłumaczenia wykonanego przez człowieka. Nie ponosimy odpowiedzialności za jakiekolwiek nieporozumienia lub błędne interpretacje wynikające z użycia tego tłumaczenia.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
